In [41]:
import warnings
warnings.filterwarnings("ignore")
from mlflow import MlflowClient, set_tracking_uri
import mlflow
from typing import Tuple
from tqdm import tqdm
import pandas as pd
from datetime import datetime, timedelta
import mysql.connector
import pyarrow
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import argparse
import os
from dateutil.relativedelta import relativedelta



def get_cutoff_indices(
    data: pd.DataFrame,
    n_features: int, 
    step_size:int
) -> list:
    
    stop_position = len(data) - 1
    
    subseq_first_idex = 0
    subseq_mid_idx = n_features
    subseq_last_idx = n_features + 1
    indices = []
    
    while subseq_last_idx <= stop_position:
        indices.append((subseq_first_idex, subseq_mid_idx, subseq_last_idx))
        
        subseq_first_idex += step_size
        subseq_mid_idx += step_size
        subseq_last_idx += step_size
        
    return indices




def transform_ts_data_into_features_and_target(
    ts_data: pd.DataFrame,
    input_seq_len: int,
    step_size: int
) -> pd.DataFrame:
    """
    Slices and transposes data from time-series format into a (features, target)
    format that we can use to train Supervised ML models
    """
    assert set(ts_data.columns) == {'datetime', 'Open', 'Exchange'}

    exchanges = ts_data['Exchange'].unique()
    #print(exchanges)
    features = pd.DataFrame()
    targets = pd.DataFrame()
    
    for exchange in tqdm(exchanges):
        
        # keep only ts data for this `location_id`
        ts_data_one_exchange = ts_data.loc[
            ts_data.Exchange == exchange, 
            ['datetime', 'Open']
        ]

        # pre-compute cutoff indices to split dataframe rows
        indices = get_cutoff_indices(
            ts_data_one_exchange,
            input_seq_len,
            step_size
        )

        # slice and transpose data into numpy arrays for features and targets
        n_examples = len(indices)
        x = np.ndarray(shape=(n_examples, input_seq_len), dtype=np.float32)
        y = np.ndarray(shape=(n_examples), dtype=np.float32)
        hours = []
        
        for i, idx in enumerate(indices):
            x[i, :] = ts_data_one_exchange.iloc[idx[0]:idx[1]]['Open'].values
            y[i] = ts_data_one_exchange.iloc[idx[1]:idx[2]]['Open'].values
            hours.append(ts_data_one_exchange.iloc[idx[1]]['datetime'])


        # numpy -> pandas
        features_one_exchange = pd.DataFrame(
            x,
            columns=[f'open_previous_{i+1}_day' for i in reversed(range(input_seq_len))]
        )
        features_one_exchange['datetime'] = hours
        features_one_exchange['exchange'] = exchange

        # numpy -> pandas
        targets_one_exchange = pd.DataFrame(y, columns=[f'target_open_next_day'])

        # concatenate results
        features = pd.concat([features, features_one_exchange])
        targets = pd.concat([targets, targets_one_exchange])

    features.reset_index(inplace=True, drop=True)
    targets.reset_index(inplace=True, drop=True)

    return features, targets['target_open_next_day']


def train_test_split(
    df: pd.DataFrame,
    cutoff_date: datetime,
    target_column_name: str,
    ) -> Tuple[pd.DataFrame, pd.Series, pd.DataFrame, pd.Series]:
    """
    """
    train_data = df[df.datetime < cutoff_date].reset_index(drop=True)
    test_data = df[df.datetime >= cutoff_date].reset_index(drop=True)

    X_train = train_data.drop(columns=[target_column_name])
    y_train = train_data[target_column_name]
    X_test = test_data.drop(columns=[target_column_name])
    y_test = test_data[target_column_name]

    return X_train, y_train, X_test, y_test




In [27]:
def ts_into_features_Daily(exchange):
    temporality = 'daily'
    
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")
    
    cursor = connection.cursor()
    cursor.execute(f"SELECT * FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'")

    results = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df_original = pd.DataFrame(results, columns=columns)
    df = df_original[['id_date', 'Open','Exchange']]
    df['datetime'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
    df = df[['datetime', 'Open', 'Exchange']]

    features, targets = transform_ts_data_into_features_and_target(
        df,
        input_seq_len=31*6, # one week of history -> 24*7*1
        step_size=31,
    )
    
    df = pd.concat([features, targets],
               axis = 1)
    
    print(df)
    
    #X_train, y_train, X_test, y_test = train_test_split(
    #    df,
    #    cutoff_date=datetime(2023, 5, 1, 0, 0, 0),
    #    target_column_name='target_open_next_day'
    #)
    
    # Calculate the cutoff_date as the first day of 6 months ago
    cutoff_date = (datetime.now() - relativedelta(months=6)).replace(day=1)
    
    print(cutoff_date)

    # Use the provided train_test_split function
    X_train, y_train, X_test, y_test = train_test_split(
        df,
        cutoff_date=cutoff_date,
        target_column_name='target_open_next_day'
    )
    
    # use only past close data
    past_close_columns = [c for c in X_train.columns if c.startswith('open_')]
    X_train_only_numeric = X_train[past_close_columns]
    X_test_only_numeric = X_test[past_close_columns]


    return X_test_only_numeric, X_train_only_numeric



In [28]:
def predict_daily(exchange):
    
    X_test_only_numeric, X_train_only_numeric = ts_into_features_Daily(exchange)
    
    mlflow.set_tracking_uri("http://localhost:5000")
    
    
    model_name = f"{exchange}_Daily_Model"
    model_version = "latest"
    model = mlflow.pyfunc.load_model(model_uri=f"models:/{model_name}/{model_version}")
    
    predictions = model.predict(pd.DataFrame(X_test_only_numeric))

    return predictions 

In [29]:
exchange = 'BTC-USD'
predictions = predict_daily(exchange)

MySQL DB Connected


100%|██████████| 1/1 [00:00<00:00, 12.70it/s]

     open_previous_186_day  open_previous_185_day  open_previous_184_day  \
0                    466.0                  457.0                  424.0   
1                    384.0                  391.0                  389.0   
2                    388.0                  374.0                  380.0   
3                    311.0                  318.0                  330.0   
4                    211.0                  213.0                  211.0   
..                     ...                    ...                    ...   
105                29169.0                28700.0                26636.0   
106                26606.0                26568.0                26533.0   
107                28522.0                28414.0                28332.0   
108                36165.0                36625.0                36586.0   
109                41348.0                42642.0                42261.0   

     open_previous_183_day  open_previous_182_day  open_previous_181_day  \
0          

In [26]:
predictions

array([40169.62 , 54632.04 , 68063.44 , 58678.1  , 74119.375, 63882.02 ],
      dtype=float32)

In [48]:
def ts_into_features_Daily(exchange):
    temporality = 'daily'
    
    connection = mysql.connector.connect(
        user = 'root',
        password = 'root',
        host = 'localhost',
        port = 3306,
        database = 'Historical_Data'
    )
    print("MySQL DB Connected")
    
    cursor = connection.cursor()
    cursor.execute(f"SELECT * FROM FT_DAILY_DATA WHERE Exchange = '{exchange}'")

    results = cursor.fetchall()
    columns = [column[0] for column in cursor.description]

    df_original = pd.DataFrame(results, columns=columns)
    df = df_original[['id_date', 'Open','Exchange']]
    df['datetime'] = pd.to_datetime(df['id_date'], format='%Y%m%d')
    df = df[['datetime', 'Open', 'Exchange']]

    features, targets = transform_ts_data_into_features_and_target(
        df,
        input_seq_len=31, # one week of history -> 24*7*1
        step_size=1,
    )
    
    df = pd.concat([features, targets],
               axis = 1)
    



    return df


In [49]:
exchange = 'BTC-USD'
df = ts_into_features_Daily(exchange)

MySQL DB Connected


100%|██████████| 1/1 [00:02<00:00,  2.77s/it]


In [50]:
df

,open_previous_31_day,open_previous_30_day,open_previous_29_day,open_previous_28_day,open_previous_27_day,open_previous_26_day,open_previous_25_day,open_previous_24_day,open_previous_23_day,open_previous_22_day,...,open_previous_7_day,open_previous_6_day,open_previous_5_day,open_previous_4_day,open_previous_3_day,open_previous_2_day,open_previous_1_day,datetime,exchange,target_open_next_day
0,466.0,457.0,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,...,361.0,363.0,378.0,392.0,401.0,395.0,383.0,2014-10-18,BTC-USD,384.0
1,457.0,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,...,363.0,378.0,392.0,401.0,395.0,383.0,384.0,2014-10-19,BTC-USD,391.0
2,424.0,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,...,378.0,392.0,401.0,395.0,383.0,384.0,391.0,2014-10-20,BTC-USD,389.0
3,395.0,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,377.0,...,392.0,401.0,395.0,383.0,384.0,391.0,389.0,2014-10-21,BTC-USD,382.0
4,408.0,399.0,402.0,436.0,423.0,411.0,404.0,399.0,377.0,376.0,...,401.0,395.0,383.0,384.0,391.0,389.0,382.0,2014-10-22,BTC-USD,386.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3557,68243.0,66748.0,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,...,58239.0,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,2024-07-14,BTC-USD,59225.0
3558,66748.0,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,...,55850.0,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,2024-07-15,BTC-USD,60815.0
3559,66007.0,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,...,56705.0,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,2024-07-16,BTC-USD,64784.0
3560,66189.0,66637.0,66491.0,65147.0,64960.0,64838.0,64114.0,64249.0,63173.0,60266.0,...,58034.0,57730.0,57341.0,57909.0,59225.0,60815.0,64784.0,2024-07-17,BTC-USD,65092.0
